In [5]:
# ============================================================
# NOTEBOOK: GIF Depth Analysis (YOLOv8 + MiDaS)
# Saves outputs to: .../GIFGIF_lucas/depth_analysis_outputs/
# ============================================================

import os
import cv2
import numpy as np
import torch
from PIL import Image, ImageSequence
from ultralytics import YOLO

# Ensure required packages are available (MiDaS depends on 'timm')
try:
    import timm
except ImportError:
    print("🔧 'timm' not found, installing...")
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "timm"])
    import timm

# ----------------------------
# CONFIG (EDIT IF NEEDED)
# ----------------------------
YOLO_WEIGHTS = r"D:\IIT\Year 4\FYP\Datasets\GIFGIF_lucas\yolov8n.pt"
BASE_DIR = r"D:\IIT\Year 4\FYP\Datasets\GIFGIF_lucas"
OUT_DIR = os.path.join(BASE_DIR, "depth_analysis_outputs")

# Choose how to pick the analysis frame
FRAME_MODE = "middle"  # "middle" or "first" or "sharpest"

# YOLO thresholds
PERSON_CONF_TH = 0.25
OBJ_CONF_TH = 0.25

# MiDaS model type: "MiDaS_small" (fast), "DPT_Hybrid" (better), "DPT_Large" (best but heavy)
MIDAS_MODEL_TYPE = "MiDaS_small"

os.makedirs(OUT_DIR, exist_ok=True)

print("✅ Config loaded")
print("YOLO:", YOLO_WEIGHTS)
print("OUT :", OUT_DIR)


✅ Config loaded
YOLO: D:\IIT\Year 4\FYP\Datasets\GIFGIF_lucas\yolov8n.pt
OUT : D:\IIT\Year 4\FYP\Datasets\GIFGIF_lucas\depth_analysis_outputs


In [6]:
# ============================================================
# UTILITIES: Load GIF + pick a representative frame
# ============================================================

def load_gif_frames(gif_path: str):
    gif = Image.open(gif_path)
    frames = [fr.convert("RGB") for fr in ImageSequence.Iterator(gif)]
    return frames

def pick_frame(frames, mode="middle"):
    if not frames:
        return None, -1

    if mode == "first":
        return frames[0], 0

    if mode == "middle":
        idx = len(frames) // 2
        return frames[idx], idx

    if mode == "sharpest":
        # Pick frame with highest Laplacian variance (sharpness heuristic)
        best_idx, best_score = 0, -1
        for i, fr in enumerate(frames):
            img = cv2.cvtColor(np.array(fr), cv2.COLOR_RGB2GRAY)
            score = cv2.Laplacian(img, cv2.CV_64F).var()
            if score > best_score:
                best_score = score
                best_idx = i
        return frames[best_idx], best_idx

    # fallback
    idx = len(frames) // 2
    return frames[idx], idx

In [7]:
# ============================================================
# LOAD MODELS: YOLO + MiDaS
# ============================================================

# YOLO
yolo = YOLO(YOLO_WEIGHTS)
print("✅ YOLO loaded")

# MiDaS
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("🚀 Device:", device)

midas = torch.hub.load("intel-isl/MiDaS", MIDAS_MODEL_TYPE)
midas.to(device)
midas.eval()

midas_transforms = torch.hub.load("intel-isl/MiDaS", "transforms")
if MIDAS_MODEL_TYPE in ["DPT_Large", "DPT_Hybrid"]:
    midas_transform = midas_transforms.dpt_transform
else:
    midas_transform = midas_transforms.small_transform

print(f"✅ MiDaS loaded: {MIDAS_MODEL_TYPE}")

✅ YOLO loaded
🚀 Device: cpu


Using cache found in C:\Users\akind/.cache\torch\hub\intel-isl_MiDaS_master


Loading weights:  None


d:\IIT\Year 4\FYP\Datasets\GIFGIF_lucas\gifgif-env\Lib\site-packages\torch\hub.py:247: UserWarning: You are about to download and run code from an untrusted repository. In a future release, this won't be allowed. To add the repository to your trusted list, change the command to load(..., trust_repo=False) and a command prompt will appear asking for an explicit confirmation of trust, or load(..., trust_repo=True), which will assume that the prompt is to be answered with 'yes'. You can also use load(..., trust_repo='check') which will only prompt for confirmation if the repo is not already trusted. This will eventually be the default behaviour
  _check_repo_is_trusted(


Downloading: "https://github.com/rwightman/gen-efficientnet-pytorch/zipball/master" to C:\Users\akind/.cache\torch\hub\master.zip
Downloading: "https://github.com/rwightman/pytorch-image-models/releases/download/v0.1-weights/tf_efficientnet_lite3-b733e338.pth" to C:\Users\akind/.cache\torch\hub\checkpoints\tf_efficientnet_lite3-b733e338.pth
Downloading: "https://github.com/isl-org/MiDaS/releases/download/v2_1/midas_v21_small_256.pt" to C:\Users\akind/.cache\torch\hub\checkpoints\midas_v21_small_256.pt


100%|██████████| 81.8M/81.8M [04:33<00:00, 314kB/s] 


✅ MiDaS loaded: MiDaS_small


Using cache found in C:\Users\akind/.cache\torch\hub\intel-isl_MiDaS_master


In [8]:
# ============================================================
# CORE: Run YOLO + MiDaS on a single RGB frame
# ============================================================

def compute_depth_map(rgb_frame_np: np.ndarray) -> np.ndarray:
    """
    Returns a raw depth map (float32) same HxW as input.
    Note: MiDaS outputs 'relative depth' (not metric).
    """
    input_batch = midas_transform(rgb_frame_np).to(device)

    with torch.no_grad():
        prediction = midas(input_batch)
        prediction = torch.nn.functional.interpolate(
            prediction.unsqueeze(1),
            size=rgb_frame_np.shape[:2],
            mode="bicubic",
            align_corners=False,
        ).squeeze()

    depth = prediction.cpu().numpy().astype(np.float32)
    return depth

def normalize_depth_for_view(depth: np.ndarray) -> np.ndarray:
    """
    Normalize depth to 0..255 for visualization.
    """
    dmin, dmax = float(depth.min()), float(depth.max())
    if dmax - dmin < 1e-6:
        return np.zeros_like(depth, dtype=np.uint8)
    norm = (depth - dmin) / (dmax - dmin)
    view = (norm * 255.0).clip(0, 255).astype(np.uint8)
    return view

def run_yolo_on_frame(pil_frame: Image.Image, conf_th: float = 0.25):
    """
    Returns YOLO result object.
    """
    results = yolo(pil_frame, verbose=False, conf=conf_th)
    return results[0]

def mean_depth_in_box(depth: np.ndarray, x1, y1, x2, y2) -> float:
    h, w = depth.shape[:2]
    x1 = max(0, min(int(x1), w-1))
    x2 = max(0, min(int(x2), w))
    y1 = max(0, min(int(y1), h-1))
    y2 = max(0, min(int(y2), h))
    if x2 <= x1 or y2 <= y1:
        return float("nan")
    return float(np.mean(depth[y1:y2, x1:x2]))

In [9]:
# ============================================================
# VISUALIZATION: Draw boxes + depth stats + save outputs
# ============================================================

def draw_detections_with_depth(rgb_bgr: np.ndarray, depth: np.ndarray, yolo_result, person_conf=0.25, obj_conf=0.25):
    """
    Draws:
    - Person boxes with IDs (person_1, person_2...)
    - Other objects boxes with IDs per class (dog_1, dog_2...)
    - Depth mean per box
    Returns:
      annotated_bgr, stats_list
    """
    img = rgb_bgr.copy()

    class_counts = {}
    stats = []

    names = yolo_result.names

    # iterate boxes
    if yolo_result.boxes is None or len(yolo_result.boxes) == 0:
        return img, stats

    for box in yolo_result.boxes:
        cls_id = int(box.cls[0])
        conf = float(box.conf[0])
        label = str(names.get(cls_id, cls_id)).lower()

        # threshold per type
        if label == "person":
            if conf < person_conf:
                continue
        else:
            if conf < obj_conf:
                continue

        x1, y1, x2, y2 = [int(v) for v in box.xyxy[0].tolist()]

        class_counts[label] = class_counts.get(label, 0) + 1
        obj_name = f"{label}_{class_counts[label]}"

        dval = mean_depth_in_box(depth, x1, y1, x2, y2)

        # Draw rectangle
        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)

        text = f"{obj_name} | conf={conf:.2f} | depth={dval:.2f}"
        # Put text above box
        ty = max(y1 - 8, 15)
        cv2.putText(img, text, (x1, ty), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)

        stats.append({
            "name": obj_name,
            "class": label,
            "conf": conf,
            "depth_mean": dval,
            "bbox": [x1, y1, x2, y2],
        })

    return img, stats

def save_outputs(out_dir: str, gif_path: str, frame_idx: int, rgb_frame: np.ndarray, depth_raw: np.ndarray, stats: list, annotated_bgr: np.ndarray):
    """
    Saves:
      - frame.png
      - depth_gray.png (0..255)
      - depth_heat.png (colormap)
      - annotated.png (boxes + depth text)
      - stats.txt (sorted by depth)
    """
    base = os.path.splitext(os.path.basename(gif_path))[0]
    tag = f"{base}_frame{frame_idx}"

    # Save original frame
    frame_bgr = cv2.cvtColor(rgb_frame, cv2.COLOR_RGB2BGR)
    cv2.imwrite(os.path.join(out_dir, f"{tag}_frame.png"), frame_bgr)

    # Save depth grayscale
    depth_view = normalize_depth_for_view(depth_raw)
    cv2.imwrite(os.path.join(out_dir, f"{tag}_depth_gray.png"), depth_view)

    # Save depth heatmap
    depth_heat = cv2.applyColorMap(depth_view, cv2.COLORMAP_INFERNO)
    cv2.imwrite(os.path.join(out_dir, f"{tag}_depth_heat.png"), depth_heat)

    # Save annotated detections
    cv2.imwrite(os.path.join(out_dir, f"{tag}_annotated.png"), annotated_bgr)

    # Save stats
    stats_sorted = sorted(stats, key=lambda x: (np.nan_to_num(x["depth_mean"], nan=-1.0)), reverse=True)
    txt_path = os.path.join(out_dir, f"{tag}_stats.txt")
    with open(txt_path, "w", encoding="utf-8") as f:
        f.write(f"GIF: {gif_path}\n")
        f.write(f"Frame index: {frame_idx}\n")
        f.write(f"MiDaS model: {MIDAS_MODEL_TYPE}\n")
        f.write(f"YOLO weights: {YOLO_WEIGHTS}\n\n")
        f.write("Detections sorted by mean depth (higher = 'farther/closer' depending on MiDaS convention)\n")
        f.write("NOTE: MiDaS depth is RELATIVE, not metric.\n\n")
        for s in stats_sorted:
            f.write(f"{s['name']:15s} class={s['class']:10s} conf={s['conf']:.2f} depth_mean={s['depth_mean']:.2f} bbox={s['bbox']}\n")

    return {
        "frame_path": os.path.join(out_dir, f"{tag}_frame.png"),
        "depth_gray_path": os.path.join(out_dir, f"{tag}_depth_gray.png"),
        "depth_heat_path": os.path.join(out_dir, f"{tag}_depth_heat.png"),
        "annotated_path": os.path.join(out_dir, f"{tag}_annotated.png"),
        "stats_path": txt_path,
    }

In [14]:
# ============================================================
# RUN: Analyze ONE GIF (set your gif_path)
# ============================================================

gif_path = r"D:\IIT\Year 4\FYP\Datasets\GIFGIF_lucas\Data\gifgif-images-v1\gifgif-images\1bYaHhGtueIqQ.gif"  # <-- CHANGE THIS
#""D:\IIT\Year 4\FYP\Datasets\GIFGIF_lucas\Data\gifgif-images-v1\gifgif-images\1bYaHhGtueIqQ.gif""

frames = load_gif_frames(gif_path)
frame, frame_idx = pick_frame(frames, mode=FRAME_MODE)

if frame is None:
    raise ValueError("No frames found in GIF.")

# Frame to numpy RGB
rgb = np.array(frame)  # RGB uint8
rgb_bgr = cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)

# Depth (MiDaS expects RGB)
depth_raw = compute_depth_map(rgb)

# YOLO detections
yolo_result = run_yolo_on_frame(frame, conf_th=min(PERSON_CONF_TH, OBJ_CONF_TH))

# Annotate + stats
annotated_bgr, stats = draw_detections_with_depth(
    rgb_bgr=rgb_bgr,
    depth=depth_raw,
    yolo_result=yolo_result,
    person_conf=PERSON_CONF_TH,
    obj_conf=OBJ_CONF_TH,
)

# Save everything
paths = save_outputs(
    out_dir=OUT_DIR,
    gif_path=gif_path,
    frame_idx=frame_idx,
    rgb_frame=rgb,
    depth_raw=depth_raw,
    stats=stats,
    annotated_bgr=annotated_bgr,
)

print(" Saved outputs:")
for k, v in paths.items():
    print(f" - {k}: {v}")

print("\nDetections:")
for s in stats:
    print(s)

 Saved outputs:
 - frame_path: D:\IIT\Year 4\FYP\Datasets\GIFGIF_lucas\depth_analysis_outputs\1bYaHhGtueIqQ_frame5_frame.png
 - depth_gray_path: D:\IIT\Year 4\FYP\Datasets\GIFGIF_lucas\depth_analysis_outputs\1bYaHhGtueIqQ_frame5_depth_gray.png
 - depth_heat_path: D:\IIT\Year 4\FYP\Datasets\GIFGIF_lucas\depth_analysis_outputs\1bYaHhGtueIqQ_frame5_depth_heat.png
 - annotated_path: D:\IIT\Year 4\FYP\Datasets\GIFGIF_lucas\depth_analysis_outputs\1bYaHhGtueIqQ_frame5_annotated.png
 - stats_path: D:\IIT\Year 4\FYP\Datasets\GIFGIF_lucas\depth_analysis_outputs\1bYaHhGtueIqQ_frame5_stats.txt

Detections:
{'name': 'person_1', 'class': 'person', 'conf': 0.7656329274177551, 'depth_mean': 570.7298583984375, 'bbox': [124, 0, 444, 209]}


In [ ]:
# ============================================================
# (OPTIONAL) Batch run: analyze multiple GIFs in the folder
# ============================================================

import glob

gif_files = glob.glob(os.path.join(BASE_DIR, "*.gif"))
print("Found GIFs:", len(gif_files))

MAX_GIFS = 10  # change or set None to run all
count = 0

for gp in gif_files:
    if MAX_GIFS is not None and count >= MAX_GIFS:
        break

    try:
        frames = load_gif_frames(gp)
        frame, frame_idx = pick_frame(frames, mode=FRAME_MODE)
        if frame is None:
            continue

        rgb = np.array(frame)
        rgb_bgr = cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)

        depth_raw = compute_depth_map(rgb)
        yolo_result = run_yolo_on_frame(frame, conf_th=min(PERSON_CONF_TH, OBJ_CONF_TH))

        annotated_bgr, stats = draw_detections_with_depth(
            rgb_bgr=rgb_bgr,
            depth=depth_raw,
            yolo_result=yolo_result,
            person_conf=PERSON_CONF_TH,
            obj_conf=OBJ_CONF_TH,
        )

        save_outputs(OUT_DIR, gp, frame_idx, rgb, depth_raw, stats, annotated_bgr)
        print(f"✅ Done: {os.path.basename(gp)}")
        count += 1

    except Exception as e:
        print(f"⚠️ Failed: {os.path.basename(gp)} | {repr(e)}")